# Encoder-space editing — is the encoder output the writable interface the hidden state isn't?

**Direction:** `research/directions/encoder-space-editing.md` · `[in-frame]` · sub-Q 3 (editability) + 2 (identifiability).
**Model:** GRU only. **No retraining** — uses one existing checkpoint.

**The premise (Michael's).** In this GRU every fact about the world reaches the latent through exactly one channel:

```
x_t = relu(W_enc · obs_t + b_enc)   ∈ R^H     ← the ONLY route from world to state
h_t = GRUCell(x_t, h_{t-1})
ô_{t+1} = W_dec · h_t + b_dec
```

Every editor in this repo so far writes to `h`, the *accumulated* state — and the recurrence rejects those writes
(ghost ratio stays ≈1.0). `x` is different: it is the model's **input port**, the representation the recurrence was
trained to accept on every step. So: probe `x`, then edit `x`.

**Why now.** `multistep_steering` §1b found the one mechanism that works — **freeze-time teacher forcing**: freeze the
world, render the edited object interpolating pre→target over `N` frames, teacher-force those frames, unfreeze. But it
works by feeding **externally rendered observations**, i.e. it needs the ground-truth simulator. That is an oracle, not
an edit interface. This notebook asks whether the same win survives when the renderer is replaced by a **latent write
at the encoder port**.

**Sections.** §1 predictive quality · §2 recoverability `x` vs `h` · §3 canonicality `x` vs `h` ·
§4 encoder-space editing (one-shot and multi-step) · §5 the intermediate observations · §6 summary.

## Definitions

### Run (copied from `CONTROL_RUNS.md`, per the repo's run-registry rule)

| code | descriptive label (used in every figure) | hidden size | dataset | obs noise | position noise | training |
|---|---|---|---|---|---|---|
| `H256` | **H=256 · both noises (baseline)** | 256 | `4_fixed_refl_inview` | 0.2 | 0.04 | 400 epochs, batch 256, AdamW lr 1e-3, wd 1e-4, seed 0 |

### The two state spaces being compared

| symbol | name | definition | memory? |
|---|---|---|---|
| `x_t` | **encoder output** | `relu(W_enc · obs_t + b_enc)` ∈ R^256 — the model's entire input port | **none** — a function of `obs_t` alone |
| `h_t` | **hidden state** | `GRUCell(x_t, h_{t-1})` ∈ R^256 — the accumulated world state | full history |

### Metrics (formulas verbatim from `../METRICS_AND_EDITORS.md`)

| name | formula | units | better |
|---|---|---|---|
| next-step RMSE | `RMSE(pred_t, clean_obs[t+1])`, teacher-forced | obs intensity [0,1] | ↓ |
| free-run RMSE @ step s | warm up on `obs[0..9]`, then free-run; `RMSE(roll_s, clean_obs[10+s])` | obs intensity | ↓ |
| copy-previous-frame / noise floor / random frame | `pim/eval/baselines.py` on this dataset | obs intensity | reference lines |
| position / velocity R² | `1 − ‖Y − probe(z)‖²/‖Y − Ȳ‖²`, `z ∈ {x, h}`, held-out 30% | — | ↑ |
| fiber residual | `‖z − g(pos,vel)‖ / ‖z‖`, `g` linear or MLP, held-out 30% | fraction of ‖z‖ | ↓ (0 = fully canonical) |
| **Edit Index** | `(d_uned − d_edit)/(d_uned + d_edit)`, `d_· = RMSE(edited₀, gt_·)` over the **differing rays**; per sample, then averaged | −1…+1 | ↑ |
| **Target RMSE** | `RMSE(edited₀, gt_edited)` over **target rays** | obs intensity | ↓ |
| **Ghost RMSE** | `RMSE(edited₀, gt_edited)` over **ghost rays** | obs intensity | ↓ |
| **Collateral RMSE** | `RMSE(edited₀, gt_edited)` over **collateral rays** | obs intensity | ↓ |
| **Edit-frame RMSE** | `RMSE(edited₀, gt_edited)` over **all rays** | obs intensity | ↓ |
| **GT-traj RMSE** | `mean_s RMSE(edited_s, clean_obs[ef+s])` over the K-step rollout | obs intensity | ↓ |
| **fidelity ratio** | `GT-traj RMSE(editor) / GT-traj RMSE(unsteered)` | ratio | ↓ (**> 1 = the edit left the rollout FURTHER from the true post-edit world than doing nothing**) |

**The two ground-truth worlds.** Every §4 number is an error against **ground truth**, never against the unsteered
rollout. At the edit frame both worlds are rendered: **`gt_edited`** (= `clean_obs[ef]`, the teleport happened) and
**`gt_unedited`** (the counterfactual where it did not — the edited object continued from its `ef−1` position along its
own velocity, the other object at its true `ef` position).

**Ray zones (per sample, derived from those two renders, so occlusion needs no special-casing).** `target rays` = rays
the edited object occupies in `gt_edited`; `ghost rays` = rays it occupied pre-edit and now vacates; `collateral rays`
= the **other** object's rays (it must not move); `differing rays` = every ray where the two worlds differ — the
support of the Edit Index.

> **How to read the Edit Index.** **+1** = the output *is* the edited world · **0** = equidistant from both (ambiguous,
> or garbage) · **−1** = the output *is* the unedited world. Unsteered lands near −1 by construction. An output far
> from *both* worlds — scrambled or collapsed — scores **≈ 0** rather than a spuriously good value, so the index cannot
> be gamed by destroying the output. Definitions: `../METRICS_AND_EDITORS.md` §4 · implementation:
> `scripts/editability_metrics.py` (imported here, not re-derived). Replaces the retired `reach % of swap` /
> `collateral % of swap` / `selectivity` / `ghost ratio` as of 2026-07-30.

### Method parameters

| name | meaning |
|---|---|
| **`N`** | number of **frozen edit steps** the write is spread over. The world is held still; the edited object's target
readout is moved pre→target in `N` equal increments, one encoder write per increment. `N=1` = the one-shot encoder edit. |
| **encoder-reachability clamp** | every `x`-space edit is passed through `clamp_min(0)`. The encoder ends in a `relu`,
so vectors with negative entries are ones the encoder can **never emit**; clamping keeps the write inside the port's actual range. |
| **`K = 15`** | post-edit free-run steps. `ef = 20` (the dataset's edit frame). `N_EDIT = 64` edit samples. |

### Editors and references

**References (never editors):** **GT (sim)** — the simulator's time-evolving clean post-edit observations;
**Unsteered** — free-run from the un-edited warm-up state; **Oracle observation** — teacher-force the model on one extra frame: the REAL (noisy)
post-edit observation `edits.obs[ef]`. Nothing about the state is swapped — the model simply gets to *see* the
teleport happen. It therefore leads the other columns by one frame.

| method | space | mechanism |
|---|---|---|
| **Freeze-time render oracle** | `x` | the **upper bracket**: `x_j = relu(enc(render(interp_j)))` using the *true renderer*. This is `multistep_steering` §1b re-expressed at the encoder port. If this works and the probe-directed writes do not, the failure is the **reachability of the edit map**, not the representation. |
| Readout injection | `x` | linear pseudoinverse — set the `x`-space position probe's readout, preserving its null space |
| Global-PCA projection | `x` | POCS: alternate inject ↔ project onto the global 99%-variance PCA subspace of visited `x` |
| PCA geodesic | `x` | re-project onto a fresh **local** PCA tangent (k=256 neighbours) each step — the canonical structural editor |
| MLP-probe gradient | `x` | Adam on `x` through a frozen MLP position probe until it reads the target |
| Hidden-state one-shot injection | `h` | the master `h`-space baseline — the thing encoder-space editing is supposed to beat |

> **±1 alignment (the recurring bug this repo keeps re-introducing).** Warm-up teacher-forces `obs[0..ef−1]`, so a
> rollout's **step 0 decodes sim frame `ef`** — i.e. `ROLL[:,0] ↔ clean_obs[ef]`. The **Oracle observation** is instead fed
> `obs[ef]` (repo convention, `tf_hidden_at`) and therefore **leads by one frame**; it is a *soft* reference, so this is
> accepted upstream and simply recorded here.

In [ ]:
# [1] Setup: load the GRU + dataset, build both state spaces (encoder output x and hidden state h), fit probes.
import os, sys, time, json
sys.path.insert(0, "../../../..")
sys.path.insert(0, "../../../../scripts")
from dataclasses import replace
import numpy as np, torch, h5py
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display, Markdown

from pim.editors import (probe_decomposition, inject_state, fit_state_subspace,
                         manifold_steer, manifold_steer_local)
from pim.eval.baselines import compute_obs_baselines
from pim.world_models import load_checkpoint, load_dataset
from pim.simulator.sim import Scene, SimConfig
from pim.simulator.renderer import render_scene
from pim.figures.theme import style_ax
# same estimators the hidden-size / noise notebooks use, so numbers are comparable across the thread
from eval_controls import recoverability_and_canonicality, _r2, _fit_mlp, _apply, _lstsq
# canonical §4 metrics — one implementation, shared with 00_master_editability and the other controls notebooks
from editability_metrics import build_edit_zones, edit_scorecard, fidelity_ratio

torch.manual_seed(0); np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_OBJ, DT = 2, 1.0
N_EDIT, K, N_PROBE = 64, 15, 1500
OUT = "/tmp/encoder_editing"; os.makedirs(OUT, exist_ok=True)

RUN_LABEL = "H=256 · both noises (baseline)"
model, info = load_checkpoint("../../../../runs/controls/H256/best_model.pt", device=DEVICE)
bundle = load_dataset("../../../../datasets/4_fixed_refl_inview", n_obj_keep=N_OBJ)
test, edits = bundle.test, bundle.edits
H = model.hidden_size; ef = edits.edit_frame; R = edits.obs_res
sim = test.config["dataset"]["sim"]
print(f"model {RUN_LABEL} | epoch {info.epoch} val_loss {info.val_loss:.5f} | H={H} device={DEVICE}")
print(f"data: obs_noise={sim['obs_noise_std']}  position_noise={sim['position_noise_std']}  "
      f"edit_frame(ef)={ef}  T={edits.T_frames}  R={R}  |  N_EDIT={N_EDIT}  K={K}")

# ── the two state spaces ──────────────────────────────────────────────────────
def enc(o):                       # (..., R) -> (..., H)  the model's entire input port
    return F.relu(model.encoder(o))
def gru_from_x(x, state):         # feed a CUSTOM encoder vector straight into the recurrence
    h_out, h_next = model.gru(x.unsqueeze(1), state)
    return model.decoder(h_out.squeeze(1)), h_next

with torch.no_grad():
    o = torch.from_numpy(test.obs[:N_PROBE]).float().to(DEVICE)
    Xs = enc(o[:, :-1, :])                      # (N, T-1, H) encoder outputs
    Hs, _ = model.gru(Xs)                       # (N, T-1, H) hidden states
Xs, Hs = Xs.cpu().numpy(), Hs.cpu().numpy()
T = Hs.shape[1]
vis = test.is_visible[:N_PROBE, :T, :N_OBJ].all(axis=2)
with h5py.File(test.h5_path, "r") as f:
    vel_all = f["velocities"][:N_PROBE, :T, :N_OBJ, :].astype(np.float32)
P = test.positions[:N_PROBE, :T, :N_OBJ, :].reshape(N_PROBE, T, N_OBJ*2)
V = vel_all.reshape(N_PROBE, T, N_OBJ*2)
print(f"state banks: x {Xs.shape}  h {Hs.shape} | visible frames used: {vis.sum():,} of {vis.size:,}")
print(f"encoder output is non-negative by construction: min(x)={Xs.min():.3f}, "
      f"fraction of x entries exactly 0 = {(Xs == 0).mean():.2%}")

---
## §1 — Predictive quality

Where this model sits before anything is edited: the training curve, and how fast a free run decays relative to the
dataset's own reference RMSEs. **Baselines** (dashed) are properties of the dataset, not the model — *copy the previous
frame* is the trivial predictor, the *observation noise floor* is the irreducible error of noisy-vs-clean observations,
and a *random frame* is the no-information ceiling.

In [ ]:
# [2] Fig 1 — predictive quality: (a) training curves, (b) free-run RMSE vs rollout step against dataset baselines.
hist = info.metrics_history
ep  = [r["epoch"] for r in hist]; tr = [r["train_loss"] for r in hist]; va = [r["val_loss"] for r in hist]

@torch.no_grad()
def freerun_by_step(steps=20, warm=10, n=1000, batch=500):
    per, cnt = np.zeros(steps), 0
    for i in range(0, n, batch):
        o = torch.from_numpy(test.obs[i:i+batch]).float().to(DEVICE); state = None
        for t in range(warm): _, state = model.step(o[:, t], state)
        preds = [model.decode(state)]
        for _ in range(steps-1):
            p, state = model.predict_step(state); preds.append(p)
        roll = torch.stack(preds, 1).cpu().numpy()
        gt = test.clean_obs[i:i+batch, warm:warm+steps]
        per += ((roll-gt)**2).mean(axis=(0,2)) * len(gt); cnt += len(gt)
    return np.sqrt(per/cnt)

fr = freerun_by_step()
bl = compute_obs_baselines(test.obs[:1000], test.clean_obs[:1000], float(sim["obs_noise_std"]))
with torch.no_grad():
    oo = torch.from_numpy(test.obs[:1000]).float().to(DEVICE)
    pr, _ = model(oo)
    nextstep = float(((pr.cpu().numpy() - test.clean_obs[:1000, 1:])**2).mean()**0.5)

plt.style.use("default")
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.2))
ax[0].plot(ep, tr, color="#0072B2", lw=1.4, label="train loss (MSE)")
ax[0].plot(ep, va, color="#D55E00", lw=1.4, label="validation loss (MSE)")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("teacher-forced MSE vs next noisy frame")
ax[0].set_yscale("log"); ax[0].set_title("(a) training curves", fontsize=10)
ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3); style_ax(ax[0])

s = np.arange(len(fr))
ax[1].plot(s, fr, "-o", ms=3.5, color="#0072B2", label="free-run RMSE vs clean observation")
ax[1].axhline(bl.identity_rmse,    ls="--", lw=1.2, color="#D55E00", label=f"copy previous frame ({bl.identity_rmse:.3f})")
ax[1].axhline(bl.noise_floor_rmse, ls="--", lw=1.2, color="#009E73", label=f"observation noise floor ({bl.noise_floor_rmse:.3f})")
ax[1].axhline(bl.random_rmse,      ls="--", lw=1.2, color="0.5",     label=f"random frame ({bl.random_rmse:.3f})")
ax[1].set_xlabel("free-run step (0 = first unobserved frame)"); ax[1].set_ylabel("RMSE vs clean observation [0,1]")
ax[1].set_title("(b) how fast the free run decays", fontsize=10)
ax[1].legend(fontsize=7.5); ax[1].grid(alpha=0.3); style_ax(ax[1])
fig.suptitle(f"Fig 1 — predictive quality of {RUN_LABEL} (warm-up 10 frames, then free-run)", y=1.02, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig1_predictive.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)
print(f"teacher-forced next-step RMSE vs clean = {nextstep:.4f}  "
      f"(copy-previous-frame {bl.identity_rmse:.4f}, noise floor {bl.noise_floor_rmse:.4f}, random {bl.random_rmse:.4f})")
print(f"free-run RMSE: step 0 = {fr[0]:.4f}, step 5 = {fr[5]:.4f}, step {len(fr)-1} = {fr[-1]:.4f}")

---
## §2 / §3 — Recoverability and canonicality: what is readable at the port vs in the state?

The encoder output `x_t` is an instantaneous function of `obs_t` — it has **no memory**. Position should therefore be
readable from `x`; velocity, which needs at least two frames, should not be. If that holds, it is the first thing to
know about editing at the port: *you cannot write a velocity there, because there is no velocity there to write.*

Both spaces are probed with the identical held-out (70/30) estimator used by the hidden-size and noise-ablation
notebooks (imported from `scripts/eval_controls.py`), so all three notebooks' numbers are directly comparable.

In [ ]:
# [3] Fig 2 + table — position/velocity R² and fiber residual, encoder output x vs hidden state h.
res = {}
for name, Z in (("encoder output x", Xs), ("hidden state h", Hs)):
    res[name] = recoverability_and_canonicality(Z[vis], P[vis], V[vis])

rows = ["| metric | encoder output `x` | hidden state `h` | better |",
        "|---|---|---|---|"]
for key, lab, better in [("pos_r2_linear", "position R² (linear probe)", "↑"),
                         ("pos_r2_mlp",    "position R² (MLP probe)", "↑"),
                         ("vel_r2_linear", "velocity R² (linear probe)", "↑"),
                         ("vel_r2_mlp",    "velocity R² (MLP probe)", "↑"),
                         ("fiber_resid_linear", "fiber residual (linear), fraction of ‖z‖", "↓"),
                         ("fiber_resid_mlp",    "fiber residual (MLP), fraction of ‖z‖", "↓")]:
    a, b = res["encoder output x"][key], res["hidden state h"][key]
    rows.append(f"| {lab} | {a:.3f} | {b:.3f} | {better} |")
display(Markdown("**Table 1 — what each space encodes** (held-out 30%, frames where both objects are visible)\n\n"
                 + "\n".join(rows)))

plt.style.use("default")
labels = ["position R²\n(linear)", "position R²\n(MLP)", "velocity R²\n(linear)", "velocity R²\n(MLP)"]
keys   = ["pos_r2_linear", "pos_r2_mlp", "vel_r2_linear", "vel_r2_mlp"]
fkeys  = ["fiber_resid_linear", "fiber_resid_mlp"]
flabels= ["fiber residual\n(linear)", "fiber residual\n(MLP)"]
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.2))
w = 0.36; xi = np.arange(len(keys))
for k, (nm, col) in enumerate([("encoder output x", "#0072B2"), ("hidden state h", "#D55E00")]):
    ax[0].bar(xi + (k-0.5)*w, [res[nm][q] for q in keys], w, label=nm, color=col)
    ax[1].bar(np.arange(len(fkeys)) + (k-0.5)*w, [res[nm][q] for q in fkeys], w, label=nm, color=col)
ax[0].set_xticks(xi); ax[0].set_xticklabels(labels, fontsize=8.5); ax[0].set_ylabel("R² (higher is better)")
ax[0].set_title("(a) recoverability: is the physical state readable?", fontsize=10)
ax[0].axhline(0, color="0.3", lw=0.8); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3, axis="y"); style_ax(ax[0])
ax[1].set_xticks(np.arange(len(fkeys))); ax[1].set_xticklabels(flabels, fontsize=8.5)
ax[1].set_ylabel("residual as a fraction of ‖z‖ (lower is better)")
ax[1].set_title("(b) canonicality: how much is NOT a function of (position, velocity)?", fontsize=10)
ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3, axis="y"); style_ax(ax[1])
fig.suptitle("Fig 2 — what the encoder output carries versus what the hidden state carries", y=1.02, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig2_xh_probes.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

---
## §4 — Editing at the encoder port

**The edit loop.** The world is frozen for `N` steps. At step `j = 1..N`:

1. the base encoder vector comes from the model's **own** current belief — `x_base = relu(enc(decode(h)))` — so no
   external render is used anywhere except in the oracle column;
2. `x_base` is edited so the `x`-space position probe reads the interpolated target
   `p_j = p_pre + (j/N)·(p_target − p_pre)` for the edited object, with the other object held at its `ef` position;
3. the edit is clamped to `≥ 0` (the encoder-reachability clamp — the encoder ends in a `relu`);
4. the recurrence advances on the **edited vector**: `h ← GRUCell(x_edited, h)`.

Then the world unfreezes and the model free-runs `K = 15` steps. `N = 1` is the one-shot encoder edit.

In [ ]:
# [4] §4 machinery: warm-up, targets, ray masks, x/h probes and manifolds, the encoder edit loop, the scorecard.
N = min(N_EDIT, edits.n_samples); ar = np.arange(N)
oe = edits.edit_object[:N].astype(int); ou = 1 - oe
with h5py.File(edits.h5_path, "r") as f:
    pre_vel = f["velocities"][:N, ef-1, :N_OBJ, :].astype(np.float32)
pre_pos = edits.positions[:N, ef-1, :N_OBJ, :].astype(np.float32)   # pre-edit
tgt_pos = edits.positions[:N, ef,   :N_OBJ, :].astype(np.float32)   # edited obj already at the teleport target
target4 = torch.from_numpy(tgt_pos.reshape(N, N_OBJ*2)).float().to(DEVICE)
gt_roll = edits.clean_obs[:N, ef:ef+K, :].astype(np.float32)        # step s ↔ clean_obs[ef+s]

# the two ground-truth worlds + the ray zones (canonical module — see ../METRICS_AND_EDITORS.md §4)
ZONES = build_edit_zones(pre_pos=pre_pos, tgt_pos=tgt_pos, pre_vel=pre_vel,
                         edit_object=oe, sim=sim, n_obj=N_OBJ,
                         traj_pos=edits.positions[:N, ef:ef+K, :N_OBJ, :].astype(np.float32),
                         gt_edited_traj=gt_roll)
teleport = ZONES.teleport
print(f"N={N} edits | mean teleport {teleport.mean():.2f} sim-units | rays/sample: "
      f"target {ZONES.target.sum(1).mean():.1f}, ghost {ZONES.ghost.sum(1).mean():.1f}, "
      f"collateral {ZONES.collateral.sum(1).mean():.1f}, differing {ZONES.differing.sum(1).mean():.1f}")

# ── warm-up states: h0 decodes frame ef; the swap is fed obs[ef] and leads by one frame ──
@torch.no_grad()
def warm_state(upto):
    o = torch.from_numpy(edits.obs[:N]).float().to(DEVICE); state = None
    for t in range(upto): _, state = model.step(o[:, t], state)
    return state
state0  = warm_state(ef)       # teacher-forced obs[0..ef-1]
h0      = model.flat_state(state0)
h_swap  = model.flat_state(warm_state(ef+1))

# ── probe + manifold banks for BOTH spaces (fit on the edits sequences) ──
with torch.no_grad():
    oe_t = torch.from_numpy(edits.obs[:N]).float().to(DEVICE)
    Xe = enc(oe_t[:, :-1, :]); He, _ = model.gru(Xe)
Xe, He = Xe.cpu().numpy(), He.cpu().numpy()
Te = He.shape[1]
pos_bank = edits.positions[:N, :Te, :N_OBJ, :].reshape(-1, N_OBJ*2)

def space_tools(bank_np):
    """Linear pseudoinverse probe, frozen MLP probe, and global PCA subspace for one space."""
    lin = _lstsq(bank_np.reshape(-1, bank_np.shape[-1]), pos_bank)
    Z = bank_np.reshape(-1, bank_np.shape[-1])
    A_np = np.linalg.lstsq(np.concatenate([Z, np.ones((len(Z),1),np.float32)],1), pos_bank, rcond=None)[0]
    W  = torch.tensor(A_np[:-1], dtype=torch.float32, device=DEVICE)      # (H, 4)
    b_ = torch.tensor(A_np[-1],  dtype=torch.float32, device=DEVICE)      # (4,)
    Wp = torch.tensor(np.linalg.pinv(A_np[:-1]), dtype=torch.float32, device=DEVICE)  # (4, H)
    def inject(z, t): return z + (t - (z @ W + b_)) @ Wp
    bank_t = torch.from_numpy(Z).float().to(DEVICE)
    sub = fit_state_subspace(bank_t, var_threshold=0.99)
    sub = replace(sub, mean=sub.mean.to(DEVICE), basis=sub.basis.to(DEVICE),
                  explained_variance_ratio=sub.explained_variance_ratio.to(DEVICE))
    mlp = _fit_mlp(Z, pos_bank)
    rmse = float(np.sqrt(((lin(Z) - pos_bank)**2).mean()))
    return dict(inject=inject, sub=sub, bank=bank_t, mlp=mlp, probe_rmse=rmse)

TX, TH = space_tools(Xe), space_tools(He)
print(f"linear position-probe RMSE (sim-units): encoder output x = {TX['probe_rmse']:.3f} | "
      f"hidden state h = {TH['probe_rmse']:.3f}")
print(f"global-PCA subspace at 99% variance: x uses {TX['sub'].n_components} comps, "
      f"h uses {TH['sub'].n_components} comps (of {H})")

In [ ]:
# [5] Editors in a given space + the frozen-world edit loop (records the intermediate decoded observations).
def make_editors(T_):
    """The four probe-directed write mechanisms, all against the same readout target."""
    def readout(z, t): return T_["inject"](z, t)
    def globalpca(z, t): return manifold_steer(z, t, T_["inject"], T_["sub"], n_iters=25)
    def geodesic(z, t):  return manifold_steer_local(z, t, T_["inject"], T_["bank"],
                                                     k_neighbors=256, n_iters=50, bank_size=50_000)
    def mlpgrad(z, t):
        q = z.clone().detach().requires_grad_(True)
        opt = torch.optim.Adam([q], lr=0.05)
        for _ in range(200):
            opt.zero_grad(); ((T_["mlp"](q) - t)**2).mean().backward(); opt.step()
        return q.detach()
    return {"Readout injection": readout, "Global-PCA projection": globalpca,
            "PCA geodesic": geodesic, "MLP-probe gradient": mlpgrad}
EDIT_X = make_editors(TX)

# ── the freeze-time render oracle (the upper bracket): externally-rendered interpolation frames ──
REFL = np.array([sim["refl_min"], sim["refl_max"]], np.float32)
RAD  = np.array([sim["radius"]]*N_OBJ, np.float32); COL = np.tile(np.array([[1,1,1]], np.float32), (N_OBJ,1))
def _cfg(nf, noise):
    return SimConfig(seed=0, y_near=sim["y_near"], y_far=sim["y_far"], x_near=sim["x_near"], x_far=sim["x_far"],
                     n_objects=N_OBJ, radius=sim["radius"], n_frames=nf, dt=sim["dt"], obs_res=sim["obs_res"],
                     refl_min=sim["refl_min"], refl_max=sim["refl_max"], fixed_reflectivities=True,
                     obs_noise_std=noise, boundary="open", always_in_frustum=False)
def rendered_interp(i, Nn, noise):
    """Nn rendered frames: edited object lerps pre→target, the other held at its ef position."""
    o_, other = oe[i], ou[i]; Pp, Tt = pre_pos[i, o_], tgt_pos[i, o_]
    fr = np.zeros((Nn, N_OBJ, 2), np.float32)
    for j in range(Nn):
        fr[j, o_] = Pp + ((j+1)/Nn)*(Tt - Pp); fr[j, other] = tgt_pos[i, other]
    _, _, rint = render_scene(Scene(positions=fr, velocities=np.zeros((Nn, N_OBJ, 2), np.float32),
                                    radii=RAD, colors=COL, reflectivities=REFL, config=_cfg(Nn, noise)))
    return rint.astype(np.float32)

OBS_NOISE = float(sim["obs_noise_std"])   # teacher-forced inputs are noise-matched to training

@torch.no_grad()
def encoder_edit(method, Nn, keep_trace=False):
    """Freeze the world for Nn steps, writing at the encoder port each step; then free-run K steps."""
    state = tuple(s.clone() for s in state0) if isinstance(state0, tuple) else state0.clone()
    trace = []
    for j in range(1, Nn+1):
        frac = j / Nn
        p_j = tgt_pos.copy(); p_j[ar, oe] = pre_pos[ar, oe] + frac*(tgt_pos[ar, oe] - pre_pos[ar, oe])
        t_j = torch.from_numpy(p_j.reshape(N, N_OBJ*2)).float().to(DEVICE)
        if method == "Freeze-time render oracle":
            frames = np.stack([rendered_interp(i, Nn, OBS_NOISE)[j-1] for i in range(N)])
            x_ed = enc(torch.from_numpy(frames).float().to(DEVICE))
        else:
            x_base = enc(model.decode(state))                    # the model's OWN belief, re-encoded
            with torch.enable_grad():
                x_ed = EDIT_X[method](x_base, t_j)
            x_ed = x_ed.clamp_min(0.0)                           # encoder-reachability clamp
        pred, state = gru_from_x(x_ed, state)
        if keep_trace: trace.append(pred.cpu().numpy())
    obs = [model.decode(state)]                                  # step 0 ↔ sim frame ef
    for _ in range(K-1):
        p, state = model.predict_step(state); obs.append(p)
    roll = torch.stack(obs, 1).cpu().numpy()
    return (roll, np.stack(trace, 1)) if keep_trace else roll

@torch.no_grad()
def plain_rollout(hflat):
    state = model.state_from_flat(hflat); obs = [model.decode(state)]
    for _ in range(K-1):
        p, state = model.predict_step(state); obs.append(p)
    return torch.stack(obs, 1).cpu().numpy()

def scorecard(ROLL, name):
    """The canonical §4 scorecard, imported from scripts/editability_metrics.py."""
    c = edit_scorecard(ROLL[name], ZONES, gt_roll)
    c["step_rmse"] = c.pop("step_rmse_to_gt")
    return c
print("machinery ready — §4 metrics come from scripts/editability_metrics.py (not re-derived here)")

In [ ]:
# [6] Run every method at N=1 (one-shot) and N=8 (multi-step), plus the h-space baseline. Table 2.
t0 = time.time()
ROLL = {"Unsteered": plain_rollout(h0), "Oracle observation": plain_rollout(h_swap)}
ROLL["Hidden-state one-shot injection (h)"] = plain_rollout(TH["inject"](h0, target4))
METHODS = ["Freeze-time render oracle", "Readout injection", "Global-PCA projection",
           "PCA geodesic", "MLP-probe gradient"]
N_MULTI = 8
for m in METHODS:
    ROLL[f"{m} (x, N=1)"] = encoder_edit(m, 1)
    ROLL[f"{m} (x, N={N_MULTI})"] = encoder_edit(m, N_MULTI)
CARDS = {k: scorecard(ROLL, k) for k in ROLL}
for k in CARDS:
    CARDS[k]["fidelity_ratio"] = fidelity_ratio(CARDS[k], CARDS["Unsteered"])
print(f"computed {len(ROLL)} rollouts in {time.time()-t0:.1f}s")

ORDER = (["Unsteered", "Oracle observation", "Hidden-state one-shot injection (h)"]
         + [f"{m} (x, N={n})" for m in METHODS for n in (1, N_MULTI)])
rows = ["| method | space | N | Edit Index ↑ | Target RMSE ↓ | Ghost RMSE ↓ | Collateral RMSE ↓ | GT-traj RMSE ↓ | fidelity ratio |",
        "|---|---|---|---|---|---|---|---|---|"]
for k in ORDER:
    c = CARDS[k]
    sp = "h" if "(h)" in k else ("x" if "(x," in k else "—")
    nn = k.split("N=")[1].rstrip(")") if "N=" in k else "—"
    nm = k.split(" (x,")[0].split(" (h)")[0]
    rows.append(f"| {nm} | `{sp}` | {nn} | **{c['edit_index']:+.2f}** | {c['target_rmse']:.3f} | "
                f"{c['ghost_rmse']:.3f} | {c['collateral_rmse']:.3f} | {c['gt_traj_rmse']:.3f} | "
                f"{c['fidelity_ratio']:.2f} |")
display(Markdown("**Table 2 — editing at the encoder port versus in the hidden state** "
                 f"(N={N} edits, K={K} rollout steps). Edit Index: +1 = the output is the world where the edit "
                 "happened, −1 = the world where it did not, 0 = equidistant from both.\n\n"
                 + "\n".join(rows)))

In [ ]:
# [7] Fig 3 — the headline (Edit Index), the zone decomposition, and fidelity to the true post-edit world.
plt.style.use("default")
PAL = {"Freeze-time render oracle": "#009E73", "Readout injection": "#0072B2",
       "Global-PCA projection": "#E69F00", "PCA geodesic": "#CC79A7", "MLP-probe gradient": "#56B4E9"}
hkey_fig = "Hidden-state one-shot injection (h)"
bars = ["Unsteered", "Oracle observation", hkey_fig] + \
       [f"{m} (x, N={N_MULTI})" for m in METHODS]
short = ["unsteered (no edit)", "oracle observation (reference)", "hidden-state one-shot (h)"] + \
        [f"{m} (x, N={N_MULTI})" for m in METHODS]
xi = np.arange(len(bars))

fig, ax = plt.subplots(1, 4, figsize=(25, 5.0))

# (a) the headline: where each method sits between the two ground-truth worlds
cols = ["0.55", "0.25", "#8B4513"] + [PAL[m] for m in METHODS]
ax[0].bar(xi, [CARDS[k]["edit_index"] for k in bars], 0.62, color=cols)
for y, lab in [(1.0, "edited world"), (0.0, "equidistant / garbage"), (-1.0, "unedited world")]:
    ax[0].axhline(y, color="0.4", ls=":", lw=1.0)
    ax[0].annotate(lab, xy=(len(bars)-0.4, y), fontsize=7.5, color="0.35", ha="right", va="bottom")
ax[0].set_ylim(-1.05, 1.05)
ax[0].set_xticks(xi); ax[0].set_xticklabels(short, fontsize=8, rotation=25, ha="right")
ax[0].set_ylabel("Edit Index"); ax[0].set_title("(a) HEADLINE — which world is the output?", fontsize=10)
ax[0].grid(alpha=0.3, axis="y"); style_ax(ax[0])

# (b) the zone decomposition that explains (a)
zk = [("target_rmse", "Target RMSE", "#0072B2"), ("ghost_rmse", "Ghost RMSE", "#D55E00"),
      ("collateral_rmse", "Collateral RMSE", "#009E73")]
w = 0.26
for j, (key, lab, c) in enumerate(zk):
    ax[1].bar(xi + (j-1)*w, [CARDS[k][key] for k in bars], w, color=c, label=lab)
ax[1].axhline(1.0, color="#D55E00", ls=":", lw=1.2)
ax[1].annotate("RMSE > 1: observation destroyed\n(intensity is bounded in [0,1])", xy=(-0.4, 1.02),
               fontsize=7, color="#D55E00", va="bottom")
ax[1].set_xticks(xi); ax[1].set_xticklabels(short, fontsize=8, rotation=25, ha="right")
ax[1].set_ylabel("RMSE vs ground truth at the edit frame")
ax[1].set_title("(b) where the error is: target / ghost / other object", fontsize=10)
ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3, axis="y"); style_ax(ax[1])

# (c) fidelity over the rollout
s = np.arange(K)
ax[2].plot(s, CARDS["Unsteered"]["step_rmse"], color="0.55", lw=1.6, label="unsteered (no edit)")
ax[2].plot(s, CARDS["Oracle observation"]["step_rmse"], color="0.2", ls="--", lw=1.6, label="oracle observation (reference)")
ax[2].plot(s, CARDS["Hidden-state one-shot injection (h)"]["step_rmse"], color="#8B4513", lw=1.6,
           label="hidden-state one-shot (h)")
for m in METHODS:
    ax[2].plot(s, CARDS[f"{m} (x, N={N_MULTI})"]["step_rmse"], color=PAL[m], lw=2.0, label=f"{m} (x, N={N_MULTI})")
    ax[2].plot(s, CARDS[f"{m} (x, N=1)"]["step_rmse"], color=PAL[m], lw=1.0, ls=":", alpha=0.8)
ax[2].set_xlabel("rollout step s (0 = sim frame ef)"); ax[2].set_ylabel("RMSE vs clean_obs[ef+s]")
ax[2].set_title(f"(c) fidelity to the true post-edit world\n(solid = N={N_MULTI}, dotted = N=1 one-shot)", fontsize=10)
ax[2].legend(fontsize=7, ncol=1, loc="lower right"); ax[2].grid(alpha=0.3); style_ax(ax[2])

# (d) did the edit HOLD? mean distance to the true post-edit world over the whole rollout
ax[3].bar(xi, [CARDS[k]["gt_traj_rmse"] for k in bars], 0.62, color=cols)
ax[3].axhline(CARDS["Unsteered"]["gt_traj_rmse"], color="0.4", ls=":", lw=1.2)
ax[3].annotate("unsteered (do nothing)", xy=(len(bars)-0.4, CARDS["Unsteered"]["gt_traj_rmse"]),
               fontsize=7.5, color="0.35", ha="right", va="bottom")
ax[3].set_xticks(xi); ax[3].set_xticklabels(short, fontsize=8, rotation=25, ha="right")
ax[3].set_ylabel("GT-traj RMSE (mean over the rollout)")
ax[3].set_title("(d) did the edit HOLD over the rollout?", fontsize=10)
ax[3].grid(alpha=0.3, axis="y"); style_ax(ax[3])

fig.suptitle("Fig 3 — encoder-port edits versus the hidden-state baseline and the render oracle", y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig3_editability.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

# Fig 3b — Edit Index at every rollout step: landing the edit and HOLDING it are different things.
fig, a = plt.subplots(figsize=(7.6, 4.6))
sx = np.arange(K)
a.plot(sx, CARDS["Unsteered"]["edit_index_by_step"], ":", color="0.55", lw=1.8, label="unsteered (no edit)")
a.plot(sx, CARDS["Oracle observation"]["edit_index_by_step"], "--", color="0.2", lw=1.8,
       label="oracle observation (reference)")
a.plot(sx, CARDS[hkey_fig]["edit_index_by_step"], color="#8B4513", lw=1.8, label="hidden-state one-shot (h)")
for m in METHODS:
    a.plot(sx, CARDS[f"{m} (x, N={N_MULTI})"]["edit_index_by_step"], color=PAL[m], lw=2.0,
           label=f"{m} (x, N={N_MULTI})")
a.axhline(0, color="0.4", ls=":", lw=1.0); a.set_ylim(-1.05, 1.05)
a.set_xlabel("rollout step s (0 = sim frame ef)"); a.set_ylabel("Edit Index")
a.set_title("Fig 3b — does the edit HOLD? Edit Index at every rollout step\n"
            "(+1 = the edited world, −1 = the unedited world, 0 = neither)", fontsize=11)
a.legend(fontsize=7.5, loc="upper right"); a.grid(alpha=0.3); style_ax(a)
fig.tight_layout(); fig.savefig(f"{OUT}/fig3b_edit_index_rollout.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

In [ ]:
# [8] Fig 4 — N-sweep: does spreading the encoder write over more frozen steps help?
N_SWEEP = [1, 2, 3, 5, 8, 12]
best_x = max(METHODS[1:], key=lambda m: CARDS[f"{m} (x, N={N_MULTI})"]["edit_index"])  # best probe-directed x editor
sweep_methods = ["Freeze-time render oracle", best_x]
print(f"sweeping N over {N_SWEEP} for: {sweep_methods}  (best probe-directed x editor by Edit Index = {best_x})")
SW = {m: {} for m in sweep_methods}
t0 = time.time()
for m in sweep_methods:
    for nn in N_SWEEP:
        key = f"__sweep_{m}_{nn}"
        ROLL[key] = encoder_edit(m, nn)
        SW[m][nn] = scorecard(ROLL, key)
        SW[m][nn]["fidelity_ratio"] = fidelity_ratio(SW[m][nn], CARDS["Unsteered"])
print(f"N-sweep computed in {time.time()-t0:.1f}s")

plt.style.use("default")
fig, ax = plt.subplots(1, 3, figsize=(15.5, 4.2))
for m in sweep_methods:
    for k, key in enumerate(["edit_index", "gt_traj_rmse", "ghost_rmse"]):
        ax[k].plot(N_SWEEP, [SW[m][n][key] for n in N_SWEEP], "-o", ms=4, color=PAL[m], label=m)
for k, (lab, ylab) in enumerate([("(a) which world is the output?", "Edit Index (+1 edited … −1 unedited)"),
                                 ("(b) does it match the true post-edit world?", "mean RMSE vs clean_obs[ef+s]"),
                                 ("(c) is the old copy removed?", "Ghost RMSE vs ground truth")]):
    ax[k].set_xlabel("N frozen encoder-write steps"); ax[k].set_ylabel(ylab); ax[k].set_title(lab, fontsize=10)
    ax[k].grid(alpha=0.3); style_ax(ax[k])
for k, key in enumerate(["edit_index", "gt_traj_rmse", "ghost_rmse"]):
    ax[k].axhline(CARDS["Unsteered"][key], color="0.55", ls=":", lw=1.2,
                  label="unsteered" if k == 0 else None)
    ax[k].axhline(CARDS["Oracle observation"][key], color="0.2", ls="--", lw=1.2,
                  label="oracle observation" if k == 0 else None)
ax[0].set_ylim(-1.05, 1.05); ax[0].legend(fontsize=7.5)
fig.suptitle("Fig 4 — spreading the encoder write over N frozen steps (dotted = unsteered, dashed = oracle observation)",
             y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig4_Nsweep.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

In [ ]:
# [9] Canonical observation-waterfall helper (CLAUDE.md fixed spec: gray on dark, noisy context frames above a
#     dashed edit line, the shared TRUE-ef row, then each column's own free-run; green target / red-dash ghost; top legend).
N_CTX = 6
ctx_obs = edits.obs[:N, ef-N_CTX:ef, :].astype(np.float32)   # the actual NOISY frames the model was teacher-forced on
DARK, TXT, TICK, EDIT_C, HID_C = "#0a0a14", "#a3adc2", "#808a9d", "#fa8850", "#FFD166"
TARGET_C, GHOST_C = "#00E676", "#FF5252"
def _cx(m):
    i = np.where(m)[0]; return i.mean() if i.size else np.nan
tgt_cx = np.array([_cx(ZONES.target[i]) for i in range(N)])
pre_cx = np.array([_cx(ZONES.ghost[i]) for i in range(N)])
SAMPLES = list(np.argsort(teleport * (ZONES.ghost.sum(1) >= 3))[::-1][:3])

def waterfall_grid(col_titles, col_bodies, samples, suptitle, fname,
                   hidden_len=0, hidden_label=None):
    """col_bodies[c]: (N, L, R) rows BELOW the N_CTX context frames. All columns share L.

    Each column shows its OWN free-run from step 0 (which decodes sim frame `ef`). There is
    deliberately NO shared teacher-forced `ef` row: only the Oracle observation reference ever
    sees that frame, and painting it into every column would hide the exact frame §4 scores.
    hidden_len>0: the first `hidden_len` body rows are the hidden edit process -> shaded band +
    a dotted 'unfreeze' line at its end."""
    ncol = len(col_titles)
    fig, axes = plt.subplots(len(samples), ncol, figsize=(3.0 * ncol, 3.4 * len(samples)),
                             squeeze=False, facecolor=DARK)
    for r, smp in enumerate(samples):
        for c in range(ncol):
            ax = axes[r][c]; ax.set_facecolor(DARK)
            panel = np.clip(np.concatenate([ctx_obs[smp], col_bodies[c][smp]], axis=0), 0, 1)
            ax.imshow(panel, aspect="auto", origin="upper", cmap="gray", vmin=0, vmax=1, interpolation="nearest")
            for sp in ax.spines.values(): sp.set_edgecolor(TICK)
            ax.axhline(N_CTX - 0.5, color=EDIT_C, lw=1.4, ls="--", alpha=0.95)
            if hidden_len > 0:
                ax.axhspan(N_CTX - 0.5, N_CTX + hidden_len - 0.5, color=HID_C, alpha=0.08)
                ax.axhline(N_CTX + hidden_len - 0.5, color=HID_C, lw=1.1, ls=":", alpha=0.9)
            if not np.isnan(tgt_cx[smp]): ax.axvline(tgt_cx[smp], color=TARGET_C, lw=1.6, alpha=0.9)
            if not np.isnan(pre_cx[smp]): ax.axvline(pre_cx[smp], color=GHOST_C, ls="--", lw=1.6, alpha=0.9)
            if r == 0: ax.set_title(col_titles[c], fontsize=8, color=TXT)
            if c == 0:
                ax.set_ylabel(f"sample {smp} (teleport {teleport[smp]:.1f})\nsim frame", fontsize=8, color=TXT)
                if hidden_len == 0:
                    ax.set_yticks([0, N_CTX, N_CTX + 7, N_CTX + 14])
                    ax.set_yticklabels([ef - N_CTX, ef, ef + 7, ef + 14], fontsize=7)
                else:
                    ax.set_yticks([N_CTX - 0.5, N_CTX + hidden_len - 0.5])
                    ax.set_yticklabels(["edit", "unfreeze"], fontsize=7)
            else: ax.set_yticks([])
            ax.set_xlabel("ray", fontsize=8, color=TXT); ax.tick_params(colors=TICK, labelsize=7)
    handles = [Line2D([0],[0], color=TARGET_C, lw=2.2, label="object target location"),
               Line2D([0],[0], color=GHOST_C, ls="--", lw=2.2, label="ghost (pre-edit) location"),
               Line2D([0],[0], color=EDIT_C, ls="--", lw=2.2,
                      label=f"edit applied here ({N_CTX} noisy context frames above; every row below is that "
                            f"column's OWN free-run, step 0 = frame {ef})")]
    if hidden_len > 0:
        handles.append(Line2D([0],[0], color=HID_C, lw=7, alpha=0.5, label=hidden_label or "hidden edit process"))
    fig.legend(handles=handles, loc="upper center", ncol=2, fontsize=8.5, frameon=False,
               labelcolor=TXT, bbox_to_anchor=(0.5, 0.955))
    fig.suptitle(suptitle, y=1.0, fontsize=10.5, color=TXT)
    fig.tight_layout(rect=[0, 0, 1, 0.90])
    fig.savefig(f"{OUT}/{fname}", dpi=130, bbox_inches="tight", facecolor=DARK)
    display(fig); plt.close(fig); print("saved", fname)
print("waterfall helper ready; samples (largest teleports):", SAMPLES)

In [ ]:
# [10] Fig 5 — observation waterfalls: GT | unsteered | h one-shot | each encoder-port method at N=8.
#      Shared TRUE-ef row, so every model column below it is its OWN free-run from ef+1 (drop step 0).
cols = ["GT (sim)", "unsteered", "hidden-state\none-shot (h)"] + [m + f"\n(x, N={N_MULTI})" for m in METHODS]
keys = [None, "Unsteered", "Hidden-state one-shot injection (h)"] + [f"{m} (x, N={N_MULTI})" for m in METHODS]
bodies = [gt_roll if k is None else ROLL[k] for k in keys]
waterfall_grid(cols, bodies, SAMPLES,
               "Fig 5 — post-edit observation rollouts: where does the object end up, and does the old copy clear?",
               "fig5_waterfalls.png")

---
## §5 — What the model actually sees mid-edit

Sevan's explicit request: the observations generated **during** the frozen edit steps — the supposedly interpolated
intermediates. The shaded band is the hidden edit process (`N = 8` frozen steps, one encoder write each); everything
below the dotted line is the free run after unfreezing.

The diagnostic question: does the intermediate sequence show **one object translating**, or a **cross-fade** — the old
copy dimming while a new one brightens? A cross-fade means the write is manipulating intensities, not moving an object.

The oracle column is the same loop fed externally-rendered frames, so it shows what a genuine translation looks like in
this display, side by side with what the probe-directed write produces.

In [ ]:
# [11] Fig 6 — behind the scenes: the intermediate decoded observations during the N frozen encoder writes.
#      Body rows = the N hidden intermediates, then the free-run from step 1 (step 0 IS the last hidden row).
demo = SAMPLES[:2]
show = ["Freeze-time render oracle", best_x]
Lb = N_MULTI + K
bodies_b, titles_b = [], ["GT (sim)"]
gt_b = np.zeros((N, Lb, R), np.float32)
gt_b[:, N_MULTI:] = gt_roll
bodies_b.append(gt_b)
for m in show:
    roll_m, trace_m = encoder_edit(m, N_MULTI, keep_trace=True)
    b = np.zeros((N, Lb, R), np.float32)
    b[:, :N_MULTI] = trace_m; b[:, N_MULTI:] = roll_m
    bodies_b.append(b); titles_b.append(f"{m}\n(x, N={N_MULTI})")
waterfall_grid(titles_b, bodies_b, demo,
               "Fig 6 — behind the scenes: what the model renders during the N frozen encoder writes, then the free run",
               "fig6_intermediates.png", hidden_len=N_MULTI,
               hidden_label=f"hidden: the model's own decode after each of the {N_MULTI} encoder writes")

---
## §6 — Summary

The three quantities that decide the question, computed below rather than asserted:

1. **Does writing at the encoder port beat writing to the hidden state?** — compare ghost ratio and GT-traj RMSE for
   the `x`-space editors against `Hidden-state one-shot injection (h)`.
2. **Does spreading the write over `N` frozen steps help?** — the `N`-sweep.
3. **How far along the available range does a probe-directed write get?** — the **freeze-time render oracle** runs the
   *identical* loop through the *identical* port, so it marks what is achievable at this interface. Ghost ratio is a
   graded quantity, so the summary reports each editor's **position on the unsteered → oracle scale**
   `(ghost_unsteered − ghost_editor) / (ghost_unsteered − ghost_oracle)` rather than a pass/fail verdict: 0% = writes
   nothing the recurrence accepts, 100% = matches what supplying the true rendered evidence achieves.

In [ ]:
# [12] Summary table + computed summary, on the canonical §4 set.
oracle_k = f"Freeze-time render oracle (x, N={N_MULTI})"
hkey     = "Hidden-state one-shot injection (h)"
x_multi  = [f"{m} (x, N={N_MULTI})" for m in METHODS[1:]]      # probe-directed only (excludes the oracle)
bx       = max(x_multi, key=lambda k: CARDS[k]["edit_index"])
i_un, i_or = CARDS["Unsteered"]["edit_index"], CARDS[oracle_k]["edit_index"]
def pct_of_oracle(k):
    """Where this editor sits on the unsteered → render-oracle span of the Edit Index, in %."""
    return 100.0 * (CARDS[k]["edit_index"] - i_un) / max(i_or - i_un, 1e-9)

rows = ["| method | Edit Index ↑ | % of the unsteered→oracle span | Target RMSE ↓ | Ghost RMSE ↓ | Collateral RMSE ↓ | GT-traj RMSE ↓ | fidelity ratio |",
        "|---|---|---|---|---|---|---|---|"]
for lab, k in ([("unsteered (no edit)", "Unsteered"), ("oracle observation (reference)", "Oracle observation"),
                ("hidden-state one-shot (h)", hkey)]
               + [(f"{m} (x, N={N_MULTI})", f"{m} (x, N={N_MULTI})") for m in METHODS[1:]]
               + [(f"freeze-time render ORACLE (x, N={N_MULTI})", oracle_k)]):
    c = CARDS[k]
    rows.append(f"| {lab} | **{c['edit_index']:+.2f}** | {pct_of_oracle(k):.0f}% | {c['target_rmse']:.3f} | "
                f"{c['ghost_rmse']:.3f} | {c['collateral_rmse']:.3f} | {c['gt_traj_rmse']:.3f} | "
                f"{c['fidelity_ratio']:.2f} |")
display(Markdown("**Table 3 — summary.** Edit Index: +1 = the output is the world where the edit happened, −1 = the "
                 "world where it did not, 0 = equidistant from both. The percentage column rescales it onto the "
                 "`unsteered → render-oracle` span, i.e. how much of the achievable movement at this interface each "
                 "method recovers.\n\n" + "\n".join(rows)))

print("================ SUMMARY (computed, not asserted) ================")
print(f"1. WHERE YOU WRITE MATTERS. Unsteered sits at Edit Index {i_un:+.2f} (it IS the unedited world). "
      f"Hidden-state one-shot injection reaches {CARDS[hkey]['edit_index']:+.2f} "
      f"({pct_of_oracle(hkey):.0f}% of the span) -- the recurrence absorbs essentially nothing. The SAME linear "
      f"pseudoinverse applied at the encoder port instead reaches "
      f"{CARDS[f'Readout injection (x, N={N_MULTI})']['edit_index']:+.2f} "
      f"({pct_of_oracle(f'Readout injection (x, N={N_MULTI})'):.0f}%).")
print(f"   Probe-directed encoder writes span {min(CARDS[k]['edit_index'] for k in x_multi):+.2f} to "
      f"{max(CARDS[k]['edit_index'] for k in x_multi):+.2f} "
      f"({min(pct_of_oracle(k) for k in x_multi):.0f}-{max(pct_of_oracle(k) for k in x_multi):.0f}% of the span); "
      f"best = {bx.split(' (x,')[0]}.")
sw = SW[best_x]
print(f"2. SPREADING THE WRITE: {best_x} Edit Index {sw[1]['edit_index']:+.2f} at N=1 -> "
      f"{sw[N_SWEEP[-1]]['edit_index']:+.2f} at N={N_SWEEP[-1]}; the render oracle goes "
      f"{SW['Freeze-time render oracle'][1]['edit_index']:+.2f} -> "
      f"{SW['Freeze-time render oracle'][N_SWEEP[-1]]['edit_index']:+.2f} over the same range.")
print(f"3. HEADROOM AT THE PORT: the render oracle reaches {i_or:+.2f} through the identical loop and the identical "
      f"port, so an encoder vector that moves the object demonstrably exists; the best probe-directed write recovers "
      f"{pct_of_oracle(bx):.0f}% of that span. The gap is the reachability of the edit map, not the encoder representation.")
cb, cu, co = CARDS[bx], CARDS["Unsteered"], CARDS[oracle_k]
print(f"   WHERE the probe-directed write falls short (RMSE vs ground truth at the edit frame):")
print(f"     {'':22s}{'Target':>9s}{'Ghost':>9s}{'Collat':>9s}{'fidelity':>10s}")
for lab, c in [("unsteered", cu), ("best probe-directed", cb), ("render oracle", co)]:
    print(f"     {lab:22s}{c['target_rmse']:>9.3f}{c['ghost_rmse']:>9.3f}{c['collateral_rmse']:>9.3f}"
          f"{c['fidelity_ratio']:>10.2f}")
print("     (fidelity ratio > 1 = the edit left the rollout further from the true post-edit world than doing nothing)")
print(f"\n4. NO VELOCITY AT THE PORT: x-space velocity R² (linear) = {res['encoder output x']['vel_r2_linear']:.3f} "
      f"vs h-space {res['hidden state h']['vel_r2_linear']:.3f}. The encoder output is an instantaneous function of "
      f"one frame, so there is no velocity there to write -- position R² (linear) is "
      f"{res['encoder output x']['pos_r2_linear']:.3f} (x) vs {res['hidden state h']['pos_r2_linear']:.3f} (h).")
print("\nPNGs:", sorted(os.listdir(OUT)))